# Step 5 — OGS model setup

**Learning objective:** Inspect the OGS process file, diffusion coefficient, output fields and Dirichlet conditions.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Explain:** one stationary diffusion process with coefficient 1.0, constant boundary values 60 and 30. The displayed values are model potentials, not calibrated kPa.

In [ ]:
import xml.etree.ElementTree as ET
project=ROOT/'models'/'excavation.prj'
tree=ET.parse(project); r=tree.getroot()
parameters={p.findtext('name'):p.findtext('value') for p in r.findall('./parameters/parameter')}
print('Parameters:',parameters)
print('Meshes:',[m.text for m in r.findall('./meshes/mesh')])
for bc in r.findall('./process_variables/process_variable/boundary_conditions/boundary_condition'):
    print('Dirichlet mesh:',bc.findtext('mesh'),'parameter:',bc.findtext('parameter'))
assert parameters['p_bc0']=='60' and parameters['p_bc1']=='30'
print('PASS: baseline configuration checked (no simulation run yet)')

**Visualize the actual model setup:** The background is the **real 2D mesh**. Red points correspond to `excavation_101` and blue points to `excavation_102`, identified using each boundary mesh's `bulk_node_ids`. The text annotations are read from the `.prj` parameters rather than invented for a schematic. This image is the **model setup**, not an OGS solution.

In [ ]:
bulk = meshio.read(ROOT/'mesh'/'excavation.vtu')
tri = np.vstack([c.data for c in bulk.cells if c.type=='triangle'])
fig, ax=plt.subplots(figsize=(12.0, 5.7))
ax.triplot(bulk.points[:,0],bulk.points[:,1],tri,color='#cbd5e1',lw=0.28)
for tag, key, color in [(101,'p_bc0','#d32f2f'),(102,'p_bc1','#1565c0')]:
    b=meshio.read(ROOT/'mesh'/f'excavation_{tag}.vtu')
    ids=np.asarray(b.point_data['bulk_node_ids'],dtype=int).reshape(-1)
    assert np.allclose(b.points,bulk.points[ids],atol=1e-7)
    ax.scatter(b.points[:,0],b.points[:,1],s=24,c=color, zorder=5,
               label=f'Dirichlet {tag}: u = {parameters[key]} ({len(ids)} nodes)')
ax.annotate('Impermeable wall\n(excluded material)',xy=(32.5,16),xytext=(39,20),
            arrowprops=dict(arrowstyle='->'),fontsize=10)
ax.set(xlim=(0,65),ylim=(0,30.5),xlabel='x [m]',ylabel='y [m]',title='OGS setup — actual boundary submeshes and potential values')
ax.set_aspect('equal');ax.legend(loc='lower center');plt.show()
print('PASS: two mapped Dirichlet surfaces match OGS project parameters')


**Interpretation:** The *initial condition* `p0=45` initializes the unknown, while Dirichlet values `p_bc0=60` and `p_bc1=30` constrain the left and right top boundary nodes. The diffusion coefficient is `1` (mathematical model). **Exercise before running:** Predict where the steepest gradients should appear and whether lowering the left boundary to 50 changes the streamline geometry. **Next:** solve the reference case.